# 2회차 실습 노트북

실습가이드의 명령을 순서대로 셀로 옮긴 노트북임. 코랩, 개인 PC, 실습 서버에서 똑같이 쓸 수 있음

- 셀은 위에서부터 차례로 Shift + Enter로 실행함
- `%%writefile` 셀은 빈칸 `____`을 채운 뒤 실행하고, 바로 아래 셀로 결과를 봄
- 연결이 끊기면 0번 셀부터 다시 실행함
- 작업을 남기려면 파일 → Drive에 사본 저장. 결과 파일은 맨 아래 셀로 내려받음

## 0. 준비
- 런타임 유형은 T4 GPU 권장(런타임 → 런타임 유형 변경). CPU로도 됨
- 아래 두 셀을 실행해 판정이 '통과'인지 봄. '확인 필요'가 나오면 TA를 부름

In [ ]:
import os, sys, shutil, subprocess, threading
from IPython.display import Image, Markdown, display

REPO = "https://github.com/elliediaz/llm26-course.git"
if os.path.isfile("../src/config.py"):          # 저장소 안에서 열었을 때
    HOME = os.path.abspath("..")
elif os.path.isdir("/workspace/course"):        # 실습 서버
    HOME = "/workspace/course"
else:                                           # 코랩
    HOME = "/content/course" if os.path.isdir("/content") else os.path.abspath("llm26-course")
    if not os.path.isdir(HOME):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, HOME], check=True)
    else:
        subprocess.run(["git", "-C", HOME, "pull", "-q"])
os.chdir(HOME)
print("작업 폴더:", os.getcwd())
os.makedirs("results/fig", exist_ok=True)


def show(*paths, width=760):
    for p in paths:
        display(Image(p, width=width)) if os.path.exists(p) else print("아직 없음:", p)


def tail(path, n=8):
    if not os.path.exists(path):
        print("아직 없음:", path)
        return
    lines = open(path, encoding="utf-8", errors="replace").read().splitlines()
    print("\n".join(lines[-n:]))


def use_ref():
    # 학습이 늦을 때 1회차 참고 결과를 복사함(내 결과는 덮어쓰지 않음)
    n = 0
    for f in sorted(os.listdir("ref/s01")):
        dst = {".json": "results", ".png": "results/fig", ".pt": "runs"}.get(os.path.splitext(f)[1])
        if dst and not os.path.exists(os.path.join(dst, f)):
            os.makedirs(dst, exist_ok=True)
            shutil.copy(os.path.join("ref/s01", f), dst)
            n += 1
    print(f"참고 결과 {n}개 복사(이미 있던 파일은 그대로 둠)")

In [ ]:
%pip install -q koreanize-matplotlib sentence-transformers
!python src/check_gpu.py

## 복습 실습 R. XOR를 직접 나눠 보기
- 단층 · 층만 쌓기 · 층 + 활성화 함수, 세 모델로 AND·OR·XOR를 학습함(약 10초)

In [ ]:
!python src/xor_demo.py
show("results/fig/xor_boundary.png", width=620)

In [ ]:
!python src/xor_demo.py --hidden 2
# !python src/xor_demo.py --hidden 2 --seed 3
# !python src/xor_demo.py --act sigmoid

확인 값
- XOR 줄: 단층 0.50, 2층·활성화 X 0.50, 2층·ReLU 1.00
- `--hidden 2`: 시작값에 따라 0.50 · 0.75 · 1.00으로 갈림. `--seed`를 바꿔 다시 실행해 봄

## 실습 D ①. 기억 길이 시험
- 숫자 여러 개 중 맨 앞 숫자를 맞힘(찍으면 0.1). 거리가 멀어질수록 어떻게 되는지 봄(코랩 CPU 약 1분 30초)

In [ ]:
!python src/seq_limits.py
show("results/fig/seq_limits.png")

## 휴식 전 걸어 두기(실습 D ②)
- 한글 금액을 숫자로 바꾸는 seq2seq 세 구조를 2분씩 학습함(약 7분)
- 쉬는 동안 탭을 닫지 않음. 도는 동안 다른 학습 셀은 실행하지 않음

In [ ]:
def _run_jobs():
    with open("results/log_seq2seq.txt", "w", encoding="utf-8") as f:
        subprocess.run([sys.executable, "src/tiny_seq2seq_train.py", "--arch", "all", "--minutes", "2"],
                       stdout=f, stderr=subprocess.STDOUT)
    open("results/bg_done.txt", "w").write("done")


if os.path.exists("results/bg_done.txt"):
    os.remove("results/bg_done.txt")
threading.Thread(target=_run_jobs, daemon=True).start()
print("뒤에서 학습 시작. 진행 확인은 다음 셀")

In [ ]:
tail("results/log_seq2seq.txt", 4)
print("\n모두 끝남" if os.path.exists("results/bg_done.txt") else "\n아직 학습 중")

## 실습 D ③④. seq2seq 세 구조의 결과
- 학습이 덜 끝났으면 `use_ref()`로 참고 결과를 받아 진행함

In [ ]:
tail("results/log_seq2seq.txt", 6)
# use_ref()

In [ ]:
!python src/viz_attention.py tiny --arch gru_attn --text "삼억 칠천이백만 오천 원"
show("results/fig/attn_tiny_gru_attn.png")

확인 값
- 17자 이상에서 `gru`는 떨어지고(코랩 CPU 약 0.5), `gru_attn`은 약 1.00
- 그림: 출력 숫자마다 입력의 어느 글자를 봤는지

## 실습 1. Self-Attention을 직접 계산하고 그려 보기
- 따라 치기 2-T1: 빈칸 5개

In [ ]:
%%writefile typing/s02/t1_self_attention.py
"""[따라 치기 2-T1] Self-Attention 직접 계산 (2회차 실습 1 ②)

실행: python typing/s02/t1_self_attention.py      (빈칸 ____을 채운 뒤)
확인: ① 점수표 s는 (토큰 수 × 토큰 수): 모든 쌍의 유사도
      ② causal mask로 대각선 위쪽(미래)이 0이 됨
      ③ 행마다 합이 1 → 각 토큰이 '어느 토큰을 볼지'를 확률로 나눠 가짐(부드러운 분류)
"""
import math

import torch

torch.manual_seed(0)
T, d = 6, 8                                      # 토큰 6개, 차원 8
x = torch.randn(1, T, d)                          # (배치, 토큰, 차원)
Wq, Wk, Wv = (torch.nn.Linear(d, d, bias=False) for _ in range(3))

q, k, v = Wq(x), Wk(x), Wv(x)
s = q @ k.____(-2, -1) / math.sqrt(d)        # (1, T, T) 모든 쌍의 유사도
mask = torch.ones(T, T).____().bool()             # 아래 삼각형만 True: 자기와 과거만 봄
s = s.____(~mask, float("-inf"))           # 미래 자리는 -inf → softmax 후 0
a = s.____(dim=-1)                             # 행마다 합 1
y = ____                                         # 가중합: 새 표현

torch.set_printoptions(precision=2, sci_mode=False)
print("주의 가중치(행: 만드는 토큰, 열: 보는 토큰)\n", a[0])
print("행 합:", a[0].sum(-1))
print("출력 모양:", tuple(y.shape))

In [ ]:
!python typing/s02/t1_self_attention.py

In [ ]:
!python src/mini_transformer.py --info

In [ ]:
!python src/viz_attention.py compare --text "결제했는데 아이템이 안 들어왔어요"
show("results/fig/attn_compare.png")

확인 값
- 가중치 첫 행 1.00, 둘째 행 0.45·0.55, 대각선 위쪽 0, 행마다 합 1
- 그림: Self는 대각선 위쪽이 빈 삼각형, Cross는 번역 결과와 입력의 대응
- 따옴표 안의 문장을 바꿔 다시 그려 봄

## 실습 2. 여러 종류의 글로 토큰 수 재기
- 처음 실행할 때 토크나이저 세 개를 내려받음(약 45초)

In [ ]:
!python src/inspect_tokenizer.py

In [ ]:
texts = ["녹스 궁 쿨 몇초임",
         "for i in range(10): print(i)",
         "2026년 11월 17일 오후 6시, 3,850원",
         "오늘 진짜 최고 ㅋㅋㅋ",
         "Transformer는 RNN보다 빠름"]
for t in texts:
    print("=" * 10, t)
    !python src/inspect_tokenizer.py --text "{t}"

확인 값
- 한국어 글자당 토큰: A.X 0.46 · Qwen3 0.75 · KURE 0.60(기본 표)
- 같은 뜻이라도 모델마다 토큰 수가 다름. 토큰 수 = 비용 · 속도 · 컨텍스트 한도

## 실습 3. 다음 토큰 분류기와 고르는 법
- 따라 치기 T3: 빈칸 5개. 처음 실행할 때 모델을 내려받아 1~2분 더 걸림

In [ ]:
%%writefile typing/s01/t3_next_token.py
"""[따라 치기 T3] 다음 토큰 분류기로서의 LLM (1회차 3교시 실습 E)

실행: python typing/s01/t3_next_token.py      (빈칸 ____을 채운 뒤)
확인: ① 출력층 크기 = 어휘 크기(약 15만) = 클래스 수
      ② 다음 토큰 확률 5개, 확률 합 = 1
      ③ 생성 = 분류를 15번 반복한 결과
"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEV == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEV)   # T4 등 BF16 미지원 GPU는 FP16

# 1) 출력층의 크기 = 고를 수 있는 '클래스'의 수
print("어휘 크기(클래스 수):", model.config.____)
print("출력층:", model.lm_head)

# 2) 분류 한 번: 마지막 위치에서 다음 토큰의 확률 분포
ids = tok("결제했는데 아이템이 안", return_tensors="pt").to(DEV)
with torch.no_grad():
    logits = model(**ids).logits                      # (1, 입력 길이, 어휘 크기)
probs = torch.____(logits[0, -1].float(), dim=-1)
top = torch.____(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"{p.item():.3f}  {tok.decode(i)!r}")
print("확률 합:", round(probs.sum().item(), 4))

# 3) 생성 = 분류를 반복: 고른 토큰을 뒤에 붙이고 다시 분류
x = ids["input_ids"]
for _ in range(15):
    with torch.no_grad():
        nxt = model(x).logits[0, -1].____()        # 가장 확률이 높은 클래스(greedy)
    x = torch.____([x, nxt.view(1, 1)], dim=1)
print(tok.decode(x[0]))

In [ ]:
!python typing/s01/t3_next_token.py

- 따라 치기 2-T2: 빈칸 3개. greedy · Temperature · Top-k · Top-p별로 30번 뽑음

In [ ]:
%%writefile typing/s02/t2_sampling.py
"""[따라 치기 2-T2] 같은 확률 분포에서 토큰을 고르는 방법 = 생성 설정 (2회차 실습 2 ⑤)

1회차 T3에서 LLM은 다음 토큰을 고르는 분류기였다. 분류 결과(확률 분포)는 하나인데,
그중 무엇을 고를지(greedy·Temperature·Top-k·Top-p)에 따라 출력이 달라진다.
실행: python typing/s02/t2_sampling.py      (빈칸 ____을 채운 뒤)
확인: Temperature가 높을수록·Top-p가 클수록 '서로 다른 토큰 수'가 늘어남. greedy는 항상 1
"""
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16 if DEV == "cuda" else torch.float32).to(DEV)
ids = tok("이번 패치에서 가장 크게 달라진 점은", return_tensors="pt").to(DEV)
with torch.no_grad():
    logits = model(**ids).logits[0, -1].float()


def pick(logits, temperature=1.0, top_k=0, top_p=1.0):
    if temperature == 0:
        return logits.argmax()                                    # greedy: 항상 1등
    probs = torch.softmax(logits ____ temperature, dim=-1)            # 온도: 분포를 날카롭게(<1) 또는 평평하게(>1)
    if top_k:
        kth = torch.topk(probs, top_k).values[-1]
        probs[probs < kth] = 0                                    # 상위 k개만 남김
    if top_p < 1.0:
        s, i = torch.sort(probs, descending=True)
        drop = s.____(0) - s >= top_p                           # 누적 확률이 p를 넘은 뒤의 후보는 버림
        probs[i[drop]] = 0
    probs = probs / probs.sum()
    return torch.____(probs, 1)[0]                         # 확률대로 하나 뽑기


torch.manual_seed(42)
for setting in [dict(temperature=0), dict(temperature=0.3), dict(temperature=1.0), dict(temperature=1.5),
                dict(temperature=1.0, top_k=5), dict(temperature=1.0, top_p=0.9)]:
    picks = [tok.decode(pick(logits.clone(), **setting)) for _ in range(30)]
    print(f"{str(setting):38s} 서로 다른 토큰 {len(set(picks)):2d}개 · 많이 나온 것 {Counter(picks).most_common(3)}")

In [ ]:
!python typing/s02/t2_sampling.py

확인 값
- 어휘 크기 151,936, 확률 합 1.0
- 30번 뽑기에서 서로 다른 토큰 수: greedy·T 0.3 1개, T 1.0 22개, T 1.5 29개, Top-k 5 4개, Top-p 0.9 20개

## 실습 F. LLM으로 영화 리뷰 감성 분류
- 1회차 LSTM과 같은 NSMC 리뷰를 학습 없이 분류함. 프롬프트 뒤에 ' 긍정'과 ' 부정'을 붙여 확률을 비교함
- 리뷰 60건, 코랩 CPU 약 1분(T4면 더 짧음)

In [ ]:
!python src/llm_sentiment.py

In [ ]:
!python src/llm_sentiment.py --shots 4
show("results/fig/llm_sentiment.png", width=560)

In [ ]:
!python src/llm_sentiment.py --text "배우는 좋은데 각본이 아쉽다" "두 번 봐도 또 울었다"
# !python src/llm_sentiment.py --shots 4 --model Qwen/Qwen3-0.6B-Base
# !python src/llm_sentiment.py --shots 8

확인 값
- 예시 0개: 정확도 0.50, 모두 '긍정'으로 답함
- 예시 4개: 약 0.68, Base 모델 예시 4개: 약 0.78(1회차 LSTM 0.80)

## 해보기 ②. 내 질문 10건으로 FAQ 검색기 재기
- 아래 셀의 질문 10건을 고침(표현 바꾸기 5, 오타 2, 은어 2, 무관 질문 1)
- `gold_faq_id`에 정답 번호를 씀(`data/G/faq.json`). 무관 질문은 `null`
- 10건 중 8건 이상 맞히면 완료

In [ ]:
%%writefile my_queries.json
[
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "오타",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "오타",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "은어",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "은어",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "무관 질문",
  "query": "여기에 FAQ와 상관없는 질문을 씀",
  "gold_faq_id": null
 }
]

In [ ]:
!python src/faq_search.py --queries my_queries.json
# !python src/faq_search.py --method tfidf --queries my_queries.json

## 결과 내려받기
- 코랩은 연결이 끝나면 파일이 사라지므로 수업을 마치기 전에 실행함

In [ ]:
shutil.make_archive("s02_results", "zip", ".", "results")
try:
    from google.colab import files
    files.download("s02_results.zip")
except ImportError:
    print("저장 위치:", os.path.abspath("s02_results.zip"))

## 문제가 생기면
| 증상 | 조치 |
|---|---|
| `No such file or directory` | 0번 셀부터 다시 실행 |
| 연결이 끊김 | 다시 연결하고 0번 셀부터 실행. 실습 D 학습은 다시 걸거나 `use_ref()` |
| GPU를 쓸 수 없다는 안내 | CPU로 그대로 진행 |
| 셀이 끝나지 않음 | 셀 왼쪽 ■를 누르고 TA를 부름 |
| `Rate limited` 문구 | 1~2분 기다리면 이어짐 |